# NB-07: WanModel End-to-End Forward Pass

## Learning Objectives
- Trace WanModel.forward from video input through patchify, N DiT blocks, Head, and unpatchify back to output
- Understand the time embedding pipeline: sinusoidal -> MLP -> t [B, dim], then SiLU+Linear -> t_mod [B, 6, dim]
- See how the patchify bug in WAN 2.2 is fixed and why the fix is needed

## Prerequisites
- **Prior notebooks:** NB-06 (DiTBlock forward pass, patchify/unpatchify mechanics, Head module)
- **Assumed concepts:** Diffusion model latent space, all Phase 1 primitives

## Concept Map
- **WanModel** is the full assembled diffusion backbone -- every sub-module was covered in prior notebooks
- The time embedding pipeline (NB-01 sinusoidal + adaLN-Zero from NB-05) produces t and t_mod for Head and DiTBlocks respectively
- WanModel is the prerequisite for VACE (Track 4) and S2V (Track 5)

## Data Flow Diagram

```
WanModel.forward -- 8-Step Data Flow
=====================================
  Input: x [B, 16, F, H, W]  (16 = noise latent channels, in_dim=16)
         timestep [B]         (diffusion timestep, e.g. 0.5)
         context  [B, L, 512] (text embeddings, text_dim=512)
         |
         +-- Step 1: time_embedding   [B] -> sinusoidal -> [B, 256] -> Linear+SiLU+Linear -> t [B, 384]
         +-- Step 2: time_projection  t [B, 384] -> SiLU+Linear -> [B, 2304] -> unflatten -> t_mod [B, 6, 384]
         +-- Step 3: text_embedding   context [B, L, 512] -> Linear+GELU+Linear -> [B, L, 384]
         +-- Step 4: patchify         x [B, 16, F, H, W] -> Conv3d -> [B, 384, F, h, w] -> flatten -> [B, S, 384]
         +-- Step 5: freqs assembly   f_freqs+h_freqs+w_freqs cat -> [S, 1, 48]
         +-- Step 6: N x DiTBlock     x [B, S, 384] (shape unchanged)
         +-- Step 7: Head             x [B, S, 384] -> adaLN + Linear -> [B, S, 64]
         +-- Step 8: unpatchify       [B, S, 64] -> rearrange -> [B, 16, F, H, W]
         |
  Output: x [B, 16, F, H, W]  (denoised output latents)
```

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here] + list(_here.parents)[:6]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
sys.modules["diffsynth"] = types.ModuleType("diffsynth")
sys.modules["diffsynth.models"] = types.ModuleType("diffsynth.models")
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# NB-07 imports
from diffsynth.models.wan_video_dit import WanModel, sinusoidal_embedding_1d
import torch
import math
from einops import rearrange

print("Setup complete.")

## 1. WanModel Constructor

WanModel is the complete diffusion transformer backbone. Its constructor wires together all the sub-modules we covered in Phase 1 and Phase 2 into a single `nn.Module`. Here is the full sub-module inventory with the reduced config we use throughout:

| Sub-module | Type | Shape/Config | Role |
|------------|------|-------------|------|
| `time_embedding` | Sequential(Linear(256, 384), SiLU, Linear(384, 384)) | [B, 256] -> [B, 384] | Sinusoidal timestep to dense embedding |
| `time_projection` | Sequential(SiLU, Linear(384, 2304)) | [B, 384] -> [B, 2304] | Produces 6 modulation params per block |
| `text_embedding` | Sequential(Linear(512, 384), GELU(approx='tanh'), Linear(384, 384)) | [B, L, 512] -> [B, L, 384] | Projects text encoder output to model dim |
| `patch_embedding` | Conv3d(16, 384, kernel=(1,2,2), stride=(1,2,2)) | [B, 16, F, H, W] -> [B, 384, F, H/2, W/2] | Video latent tokenizer |
| `blocks` | ModuleList of 2 DiTBlock instances | [B, S, 384] -> [B, S, 384] | Core repeated transformer unit (NB-06) |
| `head` | Head(dim=384, out_dim=16, patch_size=(1,2,2)) | [B, S, 384] -> [B, S, 64] | Projects tokens back to output channels |

**Note on `in_dim`:** We use `in_dim=16` for the base noise-only path. In production pipelines, `in_dim` varies depending on what gets concatenated before the model:
- `in_dim=16`: noise latent only (base text-to-video)
- `in_dim=32`: noise + reference image latent (`has_image_input=True`)
- `in_dim=48`: noise + control + reference (VACE pipeline with image reference)

This is a pipeline concern, not a model architecture concern -- WanModel treats `in_dim` as just the number of input channels to its Conv3d.

Source: `diffsynth/models/wan_video_dit.py`, line 273

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 273
# Reduced config (consistent with Phase 1 notebooks)
model = WanModel(
    dim=384, in_dim=16, ffn_dim=1024, out_dim=16,
    text_dim=512, freq_dim=256, eps=1e-6,
    patch_size=(1, 2, 2), num_heads=4, num_layers=2,
    has_image_input=False,
)

# Total parameter count
total_params = sum(p.numel() for p in model.parameters())
print(f"Demo model (2 layers): {total_params:,} parameters\n")

# Per-sub-module breakdown
sub_modules = [
    ("time_embedding", model.time_embedding),
    ("time_projection", model.time_projection),
    ("text_embedding", model.text_embedding),
    ("patch_embedding", model.patch_embedding),
    ("blocks (2x DiTBlock)", model.blocks),
    ("head", model.head),
]
print("Sub-module parameter counts:")
for name, mod in sub_modules:
    count = sum(p.numel() for p in mod.parameters())
    print(f"  {name:30s}: {count:>10,} params")

# Verify total adds up
sub_total = sum(sum(p.numel() for p in mod.parameters()) for _, mod in sub_modules)
assert sub_total == total_params, f"Sub-module sum {sub_total} != total {total_params}"
print(f"\nSum of sub-modules: {sub_total:,} (matches total)")

## 2. Time Embedding Pipeline

The time embedding pipeline converts a scalar diffusion timestep into two conditioning tensors used throughout the forward pass:

**Stage 1: Timestep -> t**
```
timestep [B] -> sinusoidal_embedding_1d(freq_dim=256, timestep) -> [B, 256]
  -> time_embedding: Linear(256, 384) -> SiLU -> Linear(384, 384)
  -> t [B, 384]
```

**Stage 2: t -> t_mod**
```
t [B, 384] -> time_projection: SiLU -> Linear(384, 384*6=2304)
  -> [B, 2304] -> .unflatten(1, (6, dim)) -> t_mod [B, 6, 384]
```

**Key distinction:** Head receives `t` (shape `[B, dim]`) with 2-parameter modulation (shift + scale). DiTBlocks receive `t_mod` (shape `[B, 6, dim]`) with 6-parameter modulation (shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp). This is why both `t` and `t_mod` are computed and kept alive throughout the forward pass.

Source: `diffsynth/models/wan_video_dit.py`, lines 365-367

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 365-367
# Step-by-step time embedding pipeline trace

timestep = torch.tensor([0.5])
print(f"Input timestep: {timestep} (shape {timestep.shape})")

# Step 1a: Sinusoidal embedding
sin_emb = sinusoidal_embedding_1d(model.freq_dim, timestep)
assert sin_emb.shape == torch.Size([1, 256])
print(f"After sinusoidal_embedding_1d: {sin_emb.shape}  (freq_dim=256)")

# Step 1b: Time embedding MLP -> t
with torch.no_grad():
    t = model.time_embedding(sin_emb.to(torch.float32))
assert t.shape == torch.Size([1, 384])
print(f"After time_embedding MLP:      {t.shape}  (dim=384)")

# Step 2: Time projection -> unflatten -> t_mod
with torch.no_grad():
    t_proj = model.time_projection(t)
assert t_proj.shape == torch.Size([1, 2304])
print(f"After time_projection:         {t_proj.shape}  (dim*6=2304)")

t_mod = t_proj.unflatten(1, (6, model.dim))
assert t_mod.shape == torch.Size([1, 6, 384])
print(f"After unflatten(1, (6, dim)):  {t_mod.shape}  (6 modulation params)")

print(f"\nKey: Head receives t {t.shape}, DiTBlocks receive t_mod {t_mod.shape}")

## 3. Text Embedding Pipeline

The text embedding pipeline projects text encoder output (from T5 or CLIP) from `text_dim` to the model's internal `dim`:

```
context [B, L, text_dim=512] -> Linear(512, 384) -> GELU(approx='tanh') -> Linear(384, 384) -> [B, L, 384]
```

This is a simple 2-layer MLP that adapts the text encoder's representation to the model dimension. The sequence length `L` (number of text tokens) is preserved -- each token is projected independently.

Source: `diffsynth/models/wan_video_dit.py`, line 368

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 368
context = torch.randn(1, 10, 512)  # [B, L=10 tokens, text_dim=512]
print(f"Input context: {context.shape}  (B=1, L=10 tokens, text_dim=512)")

with torch.no_grad():
    context_embedded = model.text_embedding(context)
assert context_embedded.shape == torch.Size([1, 10, 384])
print(f"After text_embedding: {context_embedded.shape}  (B=1, L=10 tokens, dim=384)")
print(f"\nText embedding projects each token independently: 512 -> 384")

## 4. End-to-End Forward Pass

Now we trace the complete 8-step pipeline from the data flow diagram in cell 1. This is the main section of the notebook -- it shows how every sub-module connects in sequence:

1. **time_embedding:** scalar timestep -> sinusoidal -> MLP -> `t [B, 384]`
2. **time_projection:** `t` -> SiLU+Linear -> unflatten -> `t_mod [B, 6, 384]`
3. **text_embedding:** context `[B, L, 512]` -> MLP -> `[B, L, 384]`
4. **patchify:** video `[B, 16, F, H, W]` -> Conv3d -> flatten -> `[B, S, 384]` + grid `(f, h, w)`
5. **freqs assembly:** precomputed 3D RoPE frequencies -> `[S, 1, 48]`
6. **N x DiTBlock:** `[B, S, 384]` -> shape unchanged through all blocks
7. **Head:** `[B, S, 384]` -> adaLN(shift, scale from `t`) -> Linear -> `[B, S, 64]`
8. **unpatchify:** `[B, S, 64]` -> einops rearrange -> `[B, 16, F, H, W]`

### The Patchify Bug in WAN 2.2

Before we can run the forward pass, we must fix a bug in the WAN 2.2 codebase.

**The problem:** `WanModel.patchify()` at line 340 returns only the Conv3d output tensor:
```python
def patchify(self, x, control_camera_latents_input=None):
    x = self.patch_embedding(x)
    # ... control_adapter handling ...
    return x  # Returns ONLY the tensor
```

But `forward()` at line 375 destructures the result expecting a tuple:
```python
x, (f, h, w) = self.patchify(x)  # Expects (tensor, (f, h, w))
```

This causes: `ValueError: not enough values to unpack (expected 2, got 1)`

**The correct behavior** (from WAN 2.1) includes the flatten step and grid size return:
```python
b, c, f, h, w = x.shape
x = rearrange(x, 'b c f h w -> b (f h w) c')  # Flatten spatial to sequence
return x, (f, h, w)  # Return both tensor and grid dimensions
```

This is a confirmed bug verified by diff between the WAN 2.1 and WAN 2.2 codebases. We fix it with a monkey-patch below.

Source: `diffsynth/models/wan_video_dit.py`, lines 340 (patchify) and 375 (forward call site)

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 340, 355-407
# Fix the patchify bug before calling forward
import types

def patchify_fixed(self, x, control_camera_latents_input=None):
    """Corrected patchify: Conv3d + flatten + return grid dims."""
    x = self.patch_embedding(x)
    if self.control_adapter is not None and control_camera_latents_input is not None:
        y_camera = self.control_adapter(control_camera_latents_input)
        x = [u + v for u, v in zip(x, y_camera)]
        x = x[0].unsqueeze(0)
    b, c, f, h, w = x.shape
    x = rearrange(x, 'b c f h w -> b (f h w) c')
    return x, (f, h, w)

model.patchify = types.MethodType(patchify_fixed, model)
print("Patchify monkey-patch applied.\n")

# ============================================================
# Step-by-step forward pass trace (all 8 steps)
# ============================================================

# Input tensors
x = torch.randn(1, 16, 4, 8, 8)      # [B, in_dim=16, F=4, H=8, W=8]
timestep = torch.tensor([0.5])         # scalar diffusion timestep
context = torch.randn(1, 10, 512)      # [B, L=10 text tokens, text_dim=512]
print(f"Input x:       {x.shape}")
print(f"Input timestep: {timestep}")
print(f"Input context: {context.shape}")

model.eval()
with torch.no_grad():
    # Step 1: time_embedding -- sinusoidal -> MLP -> t
    # Source: diffsynth/models/wan_video_dit.py, line 365
    t = model.time_embedding(
        sinusoidal_embedding_1d(model.freq_dim, timestep).to(x.dtype)
    )
    assert t.shape == torch.Size([1, 384])
    print(f"\nStep 1 -- time_embedding:   {timestep.shape} -> {t.shape}")

    # Step 2: time_projection -> unflatten -> t_mod
    # Source: diffsynth/models/wan_video_dit.py, line 367
    t_mod = model.time_projection(t).unflatten(1, (6, model.dim))
    assert t_mod.shape == torch.Size([1, 6, 384])
    print(f"Step 2 -- time_projection:  {t.shape} -> {t_mod.shape}")

    # Step 3: text_embedding
    # Source: diffsynth/models/wan_video_dit.py, line 368
    context_emb = model.text_embedding(context)
    assert context_emb.shape == torch.Size([1, 10, 384])
    print(f"Step 3 -- text_embedding:   {context.shape} -> {context_emb.shape}")

    # Step 4: patchify (using our monkey-patched version)
    # Source: diffsynth/models/wan_video_dit.py, line 375
    x_seq, (f, h, w) = model.patchify(x)
    assert x_seq.shape == torch.Size([1, 64, 384])
    print(f"Step 4 -- patchify:         {x.shape} -> {x_seq.shape}, grid=({f}, {h}, {w})")

    # Step 5: freqs assembly
    # Source: diffsynth/models/wan_video_dit.py, lines 377-381
    freqs = torch.cat([
        model.freqs[0][:f].view(f, 1, 1, -1).expand(f, h, w, -1),
        model.freqs[1][:h].view(1, h, 1, -1).expand(f, h, w, -1),
        model.freqs[2][:w].view(1, 1, w, -1).expand(f, h, w, -1),
    ], dim=-1).reshape(f * h * w, 1, -1)
    assert freqs.shape == torch.Size([64, 1, 48])
    print(f"Step 5 -- freqs assembly:   -> {freqs.shape}  (seq_len=64, head_dim//2=48)")

    # Step 6: N x DiTBlock (2 blocks)
    # Source: diffsynth/models/wan_video_dit.py, lines 388-404
    x_blocks = x_seq
    for i, block in enumerate(model.blocks):
        x_blocks = block(x_blocks, context_emb, t_mod, freqs)
        assert x_blocks.shape == torch.Size([1, 64, 384])
        print(f"Step 6 -- DiTBlock [{i}]:     -> {x_blocks.shape}")

    # Step 7: Head (uses t, NOT t_mod)
    # Source: diffsynth/models/wan_video_dit.py, line 406
    x_head = model.head(x_blocks, t)
    assert x_head.shape == torch.Size([1, 64, 64])
    print(f"Step 7 -- Head:             {x_blocks.shape} -> {x_head.shape}")

    # Step 8: unpatchify
    # Source: diffsynth/models/wan_video_dit.py, line 407
    output = model.unpatchify(x_head, (f, h, w))
    assert output.shape == torch.Size([1, 16, 4, 8, 8])
    print(f"Step 8 -- unpatchify:       {x_head.shape} -> {output.shape}")

print(f"\nFull forward pass: {x.shape} -> {output.shape}")
assert output.shape == x.shape, "Output shape must match input shape"
print("Output shape matches input shape -- the model is a denoiser.")

# Also verify via the model's own forward() method
with torch.no_grad():
    output_direct = model(x, timestep, context)
assert output_direct.shape == torch.Size([1, 16, 4, 8, 8])
print(f"\nDirect model(x, timestep, context): {output_direct.shape} (verified)")

### Shape Trace Summary

| Step | Operation | Input Shape | Output Shape |
|------|-----------|-------------|------------- |
| 1 | `time_embedding` | `[1]` | `[1, 384]` |
| 2 | `time_projection` | `[1, 384]` | `[1, 6, 384]` |
| 3 | `text_embedding` | `[1, 10, 512]` | `[1, 10, 384]` |
| 4 | `patchify` | `[1, 16, 4, 8, 8]` | `[1, 64, 384]` |
| 5 | freqs assembly | -- | `[64, 1, 48]` |
| 6 | 2x `DiTBlock` | `[1, 64, 384]` | `[1, 64, 384]` |
| 7 | `Head` | `[1, 64, 384]` | `[1, 64, 64]` |
| 8 | `unpatchify` | `[1, 64, 64]` | `[1, 16, 4, 8, 8]` |

**Sequence length calculation:** With input `[1, 16, 4, 8, 8]` and `patch_size=(1, 2, 2)`:
- After Conv3d: `F' = 4/1 = 4`, `H' = 8/2 = 4`, `W' = 8/2 = 4`
- Sequence length `S = F' * H' * W' = 4 * 4 * 4 = 64`

**Head output dimension:** `out_dim * prod(patch_size) = 16 * (1*2*2) = 64` -- this is the per-token output that unpatchify rearranges back to the video shape.

## 5. Optional Branches

The core forward pass above covers the base text-to-video path (`has_image_input=False`). WanModel also supports optional branches that modify the data flow. We cover two here for completeness.

### Image Reference Input (`has_image_input=True`)

When `has_image_input=True`, the model accepts a reference image to guide video generation. Two things change in the forward pass:

1. **Channel concatenation:** The reference image latent `y` is concatenated with the noise latent `x` along the channel dimension before patchify. This doubles the input channels:
   ```python
   x = torch.cat([x, y], dim=1)  # [B, 16, F, H, W] + [B, 16, F, H, W] -> [B, 32, F, H, W]
   ```
   This is why `in_dim` goes from 16 to 32 when image reference is enabled.

2. **CLIP embedding concatenation:** A CLIP image embedding is projected via `img_emb` (an MLP from CLIP dim 1280 to model dim 384) and concatenated to the text context:
   ```python
   clip_embedding = self.img_emb(clip_feature)  # [B, N, 1280] -> [B, N, 384]
   context = torch.cat([clip_embedding, context], dim=1)  # Prepend CLIP to text tokens
   ```

Source: `diffsynth/models/wan_video_dit.py`, lines 370-373

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, lines 370-373
# Demonstrate the shape changes with has_image_input=True

# When has_image_input=True:
# 1. x = torch.cat([x, y], dim=1)  -- channel concat: in_dim goes from 16 to 32
# 2. context = torch.cat([context, clip_fea], dim=1)  -- CLIP features prepended

# Demonstrate the shape change:
x_noise = torch.randn(1, 16, 4, 8, 8)   # noise latent
y_ref = torch.randn(1, 16, 4, 8, 8)     # reference image latent
x_combined = torch.cat([x_noise, y_ref], dim=1)
assert x_combined.shape == torch.Size([1, 32, 4, 8, 8])
print(f"Without image ref: {x_noise.shape}  (in_dim=16)")
print(f"With image ref:    {x_combined.shape}  (in_dim=32)")

# CLIP feature concatenation to context
text_context = torch.randn(1, 10, 384)   # text tokens (already embedded)
clip_tokens = torch.randn(1, 4, 384)     # CLIP image tokens (after img_emb MLP)
full_context = torch.cat([clip_tokens, text_context], dim=1)
assert full_context.shape == torch.Size([1, 14, 384])
print(f"\nText-only context:  {text_context.shape}  (L=10 tokens)")
print(f"With CLIP prepended: {full_context.shape}  (L=4+10=14 tokens)")

### Gradient Checkpointing

The block loop at lines 388-404 has a training/inference branch:
- `model.train()` + `use_gradient_checkpointing=True`: wraps each block in `torch.utils.checkpoint.checkpoint()`, recomputing activations during backward to save VRAM
- `model.eval()`: always runs blocks directly, regardless of the flag

In our notebooks we always use `model.eval()` with `torch.no_grad()`, so gradient checkpointing is never active. In production training with 30 blocks and full resolution, gradient checkpointing is essential to fit within GPU VRAM.

Source: `diffsynth/models/wan_video_dit.py`, lines 388-404

> Both VaceWanModel (Track 4) and WanS2VModel (Track 5) inherit from WanModel and override the forward pass with their own control mechanisms. VaceWanModel adds hint injection from a parallel encoder, while WanS2VModel adds per-frame audio conditioning and FramePack motion injection.

## Summary

### Key Takeaways
- **WanModel** composes all Phase 1 and Phase 2 primitives (RMSNorm, SelfAttention, CrossAttention, adaLN-Zero, GateModule, FFN, DiTBlock, Patchify, Head) into the complete diffusion backbone. The 8-step forward pass is the foundation for both VACE and S2V variants.
- **Time embedding pipeline** produces two distinct tensors: `t [B, dim]` for Head (2-param modulation) and `t_mod [B, 6, dim]` for DiTBlocks (6-param modulation). Confusing these causes shape errors.
- **The patchify bug** in WAN 2.2 (line 340) omits the flatten step and grid size return that `forward()` expects. This must be monkey-patched for `forward()` to execute. The fix matches WAN 2.1's correct behavior.
- **Output matches input shape:** `[B, 16, F, H, W]` in, `[B, 16, F, H, W]` out -- the model is a denoiser that predicts the clean signal from the noisy input.

### Source References
| Symbol | Location |
|--------|----------|
| WanModel.__init__ | `diffsynth/models/wan_video_dit.py`, line 273 |
| WanModel.forward | `diffsynth/models/wan_video_dit.py`, line 355 |
| time_embedding | `diffsynth/models/wan_video_dit.py`, line 365 |
| time_projection | `diffsynth/models/wan_video_dit.py`, line 367 |
| text_embedding | `diffsynth/models/wan_video_dit.py`, line 368 |
| patchify (buggy) | `diffsynth/models/wan_video_dit.py`, line 340 |
| patchify call site | `diffsynth/models/wan_video_dit.py`, line 375 |
| freqs assembly | `diffsynth/models/wan_video_dit.py`, lines 377-381 |
| block loop | `diffsynth/models/wan_video_dit.py`, lines 388-404 |
| head call | `diffsynth/models/wan_video_dit.py`, line 406 |
| unpatchify | `diffsynth/models/wan_video_dit.py`, line 407 |

## Exercises

### Exercise 1 -- Scale the block count
Increase `num_layers` from 2 to 4 and measure the forward pass time increase. You can use `import time; start = time.time()` around the forward call. The execution time should scale roughly linearly with the number of blocks. How does the parameter count change?

### Exercise 2 -- Production text dimension
Change `text_dim` from 512 to 4096 (the production value used with T5-XXL). Instantiate a new WanModel with this change and verify the model still works -- run the full forward pass with `context = torch.randn(1, 10, 4096)`. Compare the total parameter count to the 512 version. Which sub-module's parameter count changes and by how much?

### Exercise 3 -- Trace inside DiTBlock
Trace through the block loop manually: add print statements inside the step-by-step trace to show `t_mod.shape` being passed to each block. Verify that `t_mod` shape is `[B, 6, 384]` at every block -- it does not change between blocks. Then extract the six modulation parameters from one block to confirm they each have shape `[1, 1, 384]` after chunking.